## tshift/destriping/phase correction
Voltages read from the individual sites on a Neuropixels probe are routed to individual differential amplifiers (one per channel). The signals are then filtered (0.5-10kHz) and sampled through a multiplexer. The multiplexer samples the voltages on a group of 12 channels in order -- so the first channel in the group is sampled at the start of the sample time, and the final channel is sampled at the end of the sample time. (NP 1.0 probes are again special -- the multiplexer also samples the LFP on a channel, so there are 13 samples measured per cycle). 

For high frequency signals -- usually noise or artifacts, this difference in sample time is signficant. See an example in the discussion of global demux [here](https://billkarsh.github.io/SpikeGLX/help/catgt_tshift/catgt_tshift/).

Olivier Winter (IBL) proposed the following fix for this problem, which is implemented under various names in CatGT, IBL pipeline, and Spikeinterface):

+ Fourier transform the data
+ Apply the calculated phase shift for each channel (multiply by exp(-2*pi*f*shift)) where f is the frequency of that fourier component
+ Transform back

One of the reasons CatGT performs all the filtering in frequency space is to allow doing the tshift calculation at the same time.

The most important impact of doing this correction is lining up the sharp edges of fast artifacts across all chanels, so that the are removed with common median and common average referencing.

In [ ]:
import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy import signal
from pathlib import Path

In [ ]:
# matplotlib configuration.
labelsize = 14
tick_labelsize = 12
 
matplotlib.rc('axes', linewidth=1),
matplotlib.rc('legend', fontsize=12, handlelength=2)

matplotlib.rcParams['axes.labelsize'] = labelsize
matplotlib.rcParams['figure.autolayout'] = True
matplotlib.rcParams['font.size'] = 12
matplotlib.rcParams['mathtext.default'] = 'regular'
matplotlib.rcParams['xtick.direction'] = 'in'
matplotlib.rcParams['xtick.labelsize'] = tick_labelsize
matplotlib.rcParams['xtick.major.pad'] = 5
matplotlib.rcParams['xtick.major.size'] = 2
matplotlib.rcParams['xtick.major.width'] = 1
matplotlib.rcParams['xtick.top'] = True
matplotlib.rcParams['ytick.direction'] = 'in'
matplotlib.rcParams['ytick.labelsize'] = tick_labelsize
matplotlib.rcParams['ytick.major.pad'] = 5
matplotlib.rcParams['ytick.major.size'] = 2
matplotlib.rcParams['ytick.major.width'] = 1
matplotlib.rcParams['ytick.right'] = True

In [ ]:
# Get parameters to generate a simple dataset
fs = 30000  # samples per second
ts = 1/fs   # sample period in seconds
# frequencies for the signals
f1 = 300
f2 = 3000 

amp1 = 2 # arbitrary, pretend it's mV
amp2 = 1
offset = 0

upsamp = 100
fsg = upsamp*fs  # sample rate for the ground truth data -- this is the voltage "sampled" by teh multiplexer

npts = 1024 # number of sampled time points
npts_ind = np.arange(npts)

npts_gt = npts*upsamp

tpts_gt = np.arange(npts_gt)/(fs*upsamp)  # for plotting

raw_data_gt = offset + amp1*(np.cos(2*np.pi*f1*tpts_gt)) + amp2*(np.cos(2*np.pi*f2*tpts_gt))

# set up for 2 "channels": ch0 measured at the start of the sample period, ch1 measured halfway through
# the nominal time for the sample is the start, so just need to shift ch1
ch0_sample_ind = npts_ind*upsamp
ch1_sample_ind = ch0_sample_ind + np.floor(upsamp/2).astype("int")
ch0_data = raw_data_gt[ch0_sample_ind]
ch1_data = raw_data_gt[ch1_sample_ind]
tpts_sampled = (npts_ind/fs).astype("float")

plot_time = np.asarray([0.005,0.006]) # in seconds
samp_ind = np.floor(plot_time*fs).astype("int")
gt_ind = np.floor(plot_time*fsg).astype("int")

plot_range = np.arange(samp_ind[0],samp_ind[1])
plt.plot(tpts_sampled[plot_range], ch0_data[plot_range], tpts_sampled[plot_range], ch1_data[plot_range])
plot_range = np.arange(gt_ind[0],gt_ind[1])
plt.plot(tpts_gt[plot_range], raw_data_gt[plot_range], "g--")
plt.xlabel("time (sec)")
plt.legend(["ch0","ch1","input signal"])
plt.title("ch1 leads input signal")
plt.show()

## Calculating the shift for ch1
For each sine wave component, the signal at the common sample time (the start of the sample period) can be calculated from its measurement at a later time. The example has just two non-zero frequency components (or one, if you set amp1 or amp2 to zero), so it's easy to see that. Real data has many more frequency components, but still ideally in bounded range of frequencies: 0.5-10kHz (the hardware filter). To correct each frequency component, calculate the value it had at the start of the sample period. 

In [ ]:
ch1_fft = np.fft.fft(ch1_data,npts)  #returns a complex array
# calculate the frequency values for each point in the fft (they are not sequential!)
# np.fft.fftfreq provides the relative frequencies, multiply by fs for Hz
freq = fs*np.fft.fftfreq(npts)
t_diff = 0.5/fs  # time difference between ch0 and ch1 samples
phase_shifts = np.exp(-2j*np.pi*freq*t_diff)    # moves each frqeuency component by the time t_diff
ch1_fft_shifted = np.multiply(ch1_fft, phase_shifts)
ch1_shifted_real = np.real(np.fft.ifft(ch1_fft_shifted))

plot_range = np.arange(samp_ind[0],samp_ind[1])
plt.plot(tpts_sampled[plot_range], ch0_data[plot_range], tpts_sampled[plot_range], ch1_shifted_real[plot_range])
plot_range = np.arange(gt_ind[0],gt_ind[1])
plt.plot(tpts_gt[plot_range], raw_data_gt[plot_range], "g--")
plt.xlabel("time (sec)")
plt.legend(["ch0","ch1","input signal"])
plt.title("corrected ch1 matches input signal")
plt.show()

    

### Stuff to try
+ What happens to the data when the input frequency is higher than the Nyquist frequency for the sampling (e.g. fs/2)? Try setting f2 = 20000. Channels 1 and 2, which are sampled at different times during the sample period will now look quite different.
+ How does the undersampled data look after correction? If this were background, would it be removed with common average/common median referencing?
+ The probes try to avoid undersampling errors (called 'aliasing') by applying a hardware filter, which can be modeled as a single pass (causal) first order Butterworth. If that filter is applied before the sampling step (to raw_data_gt) how well does the phase correction work on teh filtered data? How does that depend on amplitude?